# Tessera Feynman A/B: scipy Nelder-Mead vs JAX-Adam const-opt

**Purpose:** measure whether switching tessera's constant optimizer from scipy Nelder-Mead (gradient-free, current default) to JAX-Adam (autodiff + Adam, available since the §1.3 ship) materially improves Feynman SR scores.

Both methods are already in `tessera.search.const_opt`. This notebook just configures a GP run with each and compares.

**Setup:** Runtime → Change runtime type → **T4 GPU** (or any GPU). JAX-Adam needs a GPU device to be faster than scipy; on CPU the scipy path will likely win on small problems.

**Expected wall-clock:** 8 equations × 2 methods × ~30-60s per run on T4 ≈ 8-15 min.

**Hypothesis:** for Feynman targets that are partial (rel ∈ [0.01, 0.20]) under Nelder-Mead — meaning the GP found near-correct structure but imprecise constants — JAX-Adam should close the gap by converging further on the constant subproblem.

## Setup

In [ ]:
# Verify we're on a GPU runtime
!nvidia-smi | head -20 || echo 'No GPU detected — switch runtime to GPU before proceeding'

In [ ]:
# Install tessera from GitHub (force-reinstall to bypass Colab caches)
!pip install --force-reinstall --no-deps -q git+https://github.com/davechendatascience/tessera.git

# Install tessera's runtime deps + JAX with CUDA 12 support
!pip install -q numpy scipy numba
!pip install -q --upgrade "jax[cuda12]"

**Restart the runtime now** (Runtime → Restart session) so Python picks up the fresh tessera install. Then re-run from this cell onward.

In [ ]:
import jax
import numpy as np
import time

print('JAX version:', jax.__version__)
print('JAX devices:', jax.devices())
print('Default device:', jax.default_backend())
# Expect to see something like cuda or gpu in the devices list

In [ ]:
from tessera.search import GP, GPConfig
import tessera
print('tessera version:', tessera.__version__)

## Feynman equation subset

Eight equations chosen to span the verdict tiers from the baseline `feynman_extended.md` run (8 exact / 15 partial / 7 failed of 30 at pop=400, gens=120):

| ID | Formula | Baseline verdict |
|---|---|---|
| I.6.20a | exp(-θ²/2) | exact (sanity) |
| I.6.20 | exp(-(θ/σ)²/2) | partial |
| I.12.2 | q₁q₂/(4πεr²) | failed |
| I.14.4 | 0.5·k·x² | partial |
| I.16.6 | (u+v)/(1+uv/c²) | partial |
| I.27.6 | d₁d₂/(d₁+d₂) | partial |
| I.43.31 | k·T/(6π·η·r) | failed |
| I.48.2 | mc²/√(1-v²/c²) | exact (sanity) |

Each sampler returns `(env_dict, y_array)`.

In [ ]:
def s_I_6_20a(n=2000, seed=0):
    rng = np.random.default_rng(seed)
    th = rng.uniform(0.5, 5.0, n)
    return {'theta': th}, np.exp(-(th**2) / 2.0)

def s_I_6_20(n=2000, seed=0):
    rng = np.random.default_rng(seed)
    th = rng.uniform(0.5, 5.0, n); sig = rng.uniform(0.5, 2.0, n)
    return {'theta': th, 'sigma': sig}, np.exp(-((th/sig)**2) / 2.0)

def s_I_12_2(n=2000, seed=0):
    rng = np.random.default_rng(seed)
    q1 = rng.uniform(1, 5, n); q2 = rng.uniform(1, 5, n)
    eps = rng.uniform(1, 5, n); r = rng.uniform(1, 5, n)
    return {'q1': q1, 'q2': q2, 'eps': eps, 'r': r}, q1*q2/(4*np.pi*eps*r**2)

def s_I_14_4(n=2000, seed=0):
    rng = np.random.default_rng(seed)
    k = rng.uniform(1, 5, n); x = rng.uniform(-5, 5, n)
    return {'k': k, 'x': x}, 0.5 * k * x**2

def s_I_16_6(n=2000, seed=0):
    rng = np.random.default_rng(seed)
    u = rng.uniform(0, 0.5, n); v = rng.uniform(0, 0.5, n); c = rng.uniform(1, 2, n)
    return {'u': u, 'v': v, 'c': c}, (u + v) / (1 + u*v/c**2)

def s_I_27_6(n=2000, seed=0):
    rng = np.random.default_rng(seed)
    d1 = rng.uniform(1, 5, n); d2 = rng.uniform(1, 5, n)
    return {'d1': d1, 'd2': d2}, d1 * d2 / (d1 + d2)

def s_I_43_31(n=2000, seed=0):
    rng = np.random.default_rng(seed)
    k = rng.uniform(1, 5, n); T = rng.uniform(1, 5, n)
    eta = rng.uniform(1, 5, n); r = rng.uniform(1, 5, n)
    return {'k': k, 'T': T, 'eta': eta, 'r': r}, k * T / (6 * np.pi * eta * r)

def s_I_48_2(n=2000, seed=0):
    rng = np.random.default_rng(seed)
    m = rng.uniform(1, 5, n); v = rng.uniform(0, 0.5, n); c = rng.uniform(1, 2, n)
    return {'m': m, 'v': v, 'c': c}, m * c**2 / np.sqrt(1 - (v/c)**2)

SUBSET = [
    ('I.6.20a', 'exp(-theta^2/2)', s_I_6_20a, 'exact'),
    ('I.6.20',  'exp(-(theta/sigma)^2/2)', s_I_6_20, 'partial'),
    ('I.12.2',  'q1*q2/(4*pi*eps*r^2)', s_I_12_2, 'failed'),
    ('I.14.4',  '0.5*k*x^2', s_I_14_4, 'partial'),
    ('I.16.6',  '(u+v)/(1+u*v/c^2)', s_I_16_6, 'partial'),
    ('I.27.6',  'd1*d2/(d1+d2)', s_I_27_6, 'partial'),
    ('I.43.31', 'k*T/(6*pi*eta*r)', s_I_43_31, 'failed'),
    ('I.48.2',  'm*c^2/sqrt(1-v^2/c^2)', s_I_48_2, 'exact'),
]
print(f'Loaded {len(SUBSET)} Feynman equations')

## A/B runner

For each equation: run the GP with `optimize_constants_method='Nelder-Mead'` (scipy, current default) and `'jax_adam'` (JAX autodiff), same seed and other config. Record `best_rel` for each.

In [ ]:
# Tunable params — adjust here for quicker / more thorough runs
POP_SIZE = 200      # Default Feynman benchmark uses 400; 200 is a Colab-friendly mid-point
N_GENS = 60         # Default 120; 60 still reasonable for the easier subset members
SEED = 2026

def classify_verdict(rel: float) -> str:
    if rel < 0.01: return 'exact'
    if rel < 0.20: return 'partial'
    return 'failed'

def run_with_method(name, sampler, method):
    env, y = sampler()
    feature_names = list(env.keys())
    var_y = float(np.var(y))
    use_jax = (method == 'jax_adam')
    cfg = GPConfig(
        pop_size=POP_SIZE, n_gens=N_GENS,
        init_max_depth=5,
        parsimony=max(var_y * 0.005, 1e-4),
        early_stop_patience=25, seed=SEED,
        pointwise_only=True, verbose=False,
        optimize_constants_every=3,
        optimize_constants_method=method,
        optimize_constants_maxiter=50 if use_jax else 30,
        use_jax_population_eval=use_jax,
    )
    gp = GP(cfg)
    t0 = time.time()
    front = gp.run(env, y, feature_names=feature_names)
    runtime = time.time() - t0
    best = min(front, key=lambda c: c.train_loss)
    rel = best.train_loss / var_y if var_y > 0 else float('nan')
    return {
        'name': name, 'method': method, 'runtime': runtime,
        'best_cx': best.complexity, 'best_loss': best.train_loss,
        'best_rel': rel, 'best_tree': str(best.tree),
        'verdict': classify_verdict(rel),
    }

## Run the A/B

In [ ]:
pairs = []
t_start = time.time()
for i, (name, formula, sampler, baseline) in enumerate(SUBSET):
    print(f'\n[{i+1}/{len(SUBSET)}] {name}: {formula}  (baseline: {baseline})')
    try:
        r_nm = run_with_method(name, sampler, 'Nelder-Mead')
        print(f'  NM:  cx={r_nm["best_cx"]:3d} rel={r_nm["best_rel"]:.4f} ({r_nm["verdict"]}) in {r_nm["runtime"]:.1f}s')
    except Exception as e:
        print(f'  NM FAILED: {e}')
        r_nm = {'name': name, 'method': 'Nelder-Mead', 'best_rel': float("inf"), 'best_cx': -1, 'runtime': 0, 'verdict': 'error', 'error': str(e)}
    try:
        r_jax = run_with_method(name, sampler, 'jax_adam')
        print(f'  JAX: cx={r_jax["best_cx"]:3d} rel={r_jax["best_rel"]:.4f} ({r_jax["verdict"]}) in {r_jax["runtime"]:.1f}s')
    except Exception as e:
        print(f'  JAX FAILED: {e}')
        r_jax = {'name': name, 'method': 'jax_adam', 'best_rel': float("inf"), 'best_cx': -1, 'runtime': 0, 'verdict': 'error', 'error': str(e)}
    pairs.append((r_nm, r_jax, baseline))

print(f'\nTotal wall-clock: {time.time() - t_start:.1f}s')

## Results

In [ ]:
import pandas as pd
from collections import Counter

rows = []
for nm, jax_, baseline in pairs:
    rows.append({
        'Eq': nm.get('name'),
        'Baseline': baseline,
        'NM rel': nm.get('best_rel'),
        'NM verdict': nm.get('verdict'),
        'JAX rel': jax_.get('best_rel'),
        'JAX verdict': jax_.get('verdict'),
        'NM time (s)': round(nm.get('runtime', 0), 1),
        'JAX time (s)': round(jax_.get('runtime', 0), 1),
    })
df = pd.DataFrame(rows)
df

In [ ]:
# Tally + transitions
def tally(verdicts):
    c = Counter(verdicts)
    return {'exact': c.get('exact', 0), 'partial': c.get('partial', 0), 'failed': c.get('failed', 0), 'error': c.get('error', 0)}

nm_t = tally(r[0]['verdict'] for r in pairs)
jax_t = tally(r[1]['verdict'] for r in pairs)
print('Nelder-Mead:', nm_t)
print('JAX-Adam:  ', jax_t)

moves = Counter()
for nm, jax_, _ in pairs:
    moves[f'{nm["verdict"]} -> {jax_["verdict"]}'] += 1
print('\nTransitions (NM -> JAX):')
for k, v in moves.most_common():
    print(f'  {k}: {v}')

## Verdict

Look for:

- **`partial → exact` transitions**: const-opt was the bottleneck; the GP found near-correct structure and JAX-Adam refined the constants. This is the score-moving outcome.
- **`failed → partial` transitions**: const-opt closed enough of the loss gap to escape the failure tier even though the structure was still imperfect.
- **No regressions**: ideally no exact→partial or partial→failed.

If JAX-Adam wins on the exact tally by ≥ 2 equations and there are no regressions, the policy change is clear: make `jax_adam` the Feynman default and update `benchmarks/results/feynman_extended.md` accordingly.

If results are tied or noisy, the next step is to run the full 30-equation subset at `pop=400, gens=120` (matches the original baseline) for a more decisive comparison.

## Tunables — where to adjust if results need refinement

- `POP_SIZE` and `N_GENS` above: bump up if results look noisy.
- `optimize_constants_maxiter`: more iters per const-opt round; main lever for JAX-Adam.
- `optimize_constants_jax_lr`: Adam learning rate (default 1e-2). Tune up for faster convergence on smooth losses; tune down if losses diverge.
- `optimize_constants_every`: more frequent polishing (every 1 gen vs default 3) — closer to PySR's policy.